# 27B を IQ3_XXS にした 1 設定の評価（life #24・2026-11-02 の協議・Colab T4 用）

「賢いモデルをそのまま小さくする」研究の最初の手。**Qwen3.8-27B を unsloth の UD-IQ3_XXS（10.18 GiB）にした 1 設定だけ**を、local-rag の 12GB 枠の線（`docs/v2-light-model-plan.md` §1）で測る。線は研究用に緩めない。

**測る前に固定した事**（life #24 のコメントと同じ。後から救済の設定を足さない）:

- 重み: `unsloth/Qwen3.8-27B-GGUF` の `Qwen3.8-27B-UD-IQ3_XXS.gguf`（revision `4ca7207`）。画像の部品（mmproj）は載せない
- 実行系: llama.cpp b11836 の配布版 llama-server（`-ngl 999 --jinja --reasoning off -np 1 -v`）。今までの候補と同じ版・同じ引数。KV の量子化・文脈の縮小・CPU への退避などは足さない（`EXTRA_ARGS = []`）
- 思考 OFF（`--reasoning off`）。流す前の 1 問で考える文 0 文字・考える印なしを確かめ、通らなければ止める
- 出力の上限 1,024 トークン・温度 0.1（`backend/app/config.py` のまま）
- 文脈長 = ceil((62 問の入力の最大 + 1,024) / 512) × 512（`LLM_MAX_TOKENS=1` で 62 問を流して測る）
- 検索: e5-large・リランク off・C（数を比べるコード）OFF。今までの Step 2 と同じ索引の作り方
- 予算は 12GB（11,776 MiB）だけ。重みだけで 8GB の予算を超えるので 8GB は測らない

**止める条件**（どれか 1 つで打ち切り。安く落とせる順に見る）:

1. 容量: `nvidia-smi` の 1 秒ごとの最大 > 11,776 MiB、またはメモリ不足の記録
2. 速さ（50 問）: 完了の中央値 > 15.0 秒、または最初の文字の中央値 > 5.0 秒（質問を受けた時から・検索込み・T4 の測定値のまま）。線を超えた問が 26 問に達したら、その時点で中央値の超過が決まるので流すのを止める。**速さで落ちたら品質は評価しない**（回答の中身は出さない）
3. 拒否 16 問（M8）が 1 問でも外れる／M7 < 1.00／M6 < 0.80
4. 数値 12 問（N）が 1 問でも外れる（外れた時点で止める）
5. 権限・障害試験（G0）に落ちる
6. 根拠採点（M4-S・人手）が完了しない（ノートの外）

**準備の失敗と品質の不合格は分けて記録する**: 各段の結果は `STAGE_JSON` の行に `pass` / `fail`（線で落ちた）/ `prep_failure`（GPU が取れない・起動しない・切断など、線とは別の失敗）で出る。準備の失敗だけやり直してよい。最後まで流れた回を、数を見た後に流し直さない。

全部の線を通ったら、新しいランタイムで同じノートをもう 1 回流す（最終判定は別々に起動した 2 回の両方合格）。

使い方: ランタイムを **T4 GPU** にして「すべてのセルを実行」。止まるのは準備の失敗だけで、線で落ちた時は以降の段を飛ばしてまとめまで進む。

In [ ]:
# RAG 側の仮想環境（step2_measure_llamacpp.ipynb と同じ）。llama-server は配布版を使うので vLLM は入れない
!pip -q install uv
!git clone -q --depth 1 -b research/27b-iq3-speed https://github.com/Maaa2005/local-rag-agent.git /content/repo
!git -C /content/repo log -1 --format='repo HEAD %H %s'
!uv venv -q /content/rag-env
# torch は CPU 版を先に入れる（後で CPU の index を足すと uv が他の依存も引きに行く）
!uv pip install -q --python /content/rag-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/rag-env/bin/python "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml

In [ ]:
# 設定（測る前に固定・life #24 のコメントと同じ。測った後に変えない）
MODEL = "Qwen/Qwen3.8-27B"
GGUF_REPO = "unsloth/Qwen3.8-27B-GGUF"
GGUF_REV = "4ca720788d1e01f1bff70c033e0d0028fd02e502"
GGUF_FILE = "Qwen3.8-27B-UD-IQ3_XXS.gguf"
QUANT = "UD-IQ3_XXS"
LLAMA_TAG = "b11836"
LLAMA_ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_ASSET = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
EXTRA_ARGS = []                           # 救済の設定は足さない
NUMERIC_CHECK = "false"                   # C は OFF（モデルそのものの判断を見る）
BUDGET = "12GB"                           # 12GB 枠だけ（11,776 MiB）
DATASET_50 = "evaluation/dataset.jsonl"
NUMERIC_12 = "evaluation/v2/numeric_boundary.jsonl"
MEASURE_CTX = 8192                        # 入力トークン数を測る起動（測る前なので長めに取る）
FAULT_TIMEOUT = 15                        # G0 の止める試験で待つ秒
QDRANT_VER = "v1.12.6"                    # qdrant-client 1.12.1 に合わせる
REPO = "/content/repo"
RAG_PY = "/content/rag-env/bin/python"
OUT = "/content/r27b"                     # 結果の置き場所（やり直す時はセルを流す前にここを消すか名前を替える）
PORT = 8001
WATCH_EVERY = 5                           # 途中の打ち切りを見る間隔（秒）
STEP_NOTE = f"life #24 研究: {MODEL} {QUANT}・思考 OFF・C OFF・12GB だけ（llama.cpp {LLAMA_TAG}）"

In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, threading, time, urllib.request

sys.path.insert(0, REPO)   # evaluation.colab.step05 / speed_gate を使う（標準ライブラリだけなので kernel から import できる）
from evaluation.colab import step05 as s5
from evaluation.colab import speed_gate as sg

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）。消すか設定セルの OUT を替える")
OUTP.mkdir(parents=True, exist_ok=True)

STAGES = []    # 段ごとの結果。pass / fail（線で落ちた）/ prep_failure（準備の失敗）/ skipped
STOP = None    # 最初に落ちた段。以降の段は流さない
def stage(name, result, reason=None, **nums):
    global STOP
    rec = {"stage": name, "result": result, "reason": reason, **nums}
    STAGES.append(rec)
    if result in ("fail", "prep_failure") and STOP is None:
        STOP = rec
    print("STAGE_JSON " + json.dumps(rec, ensure_ascii=False))
    return rec

def prep_fail(name, reason):
    stage(name, "prep_failure", reason)
    raise RuntimeError(f"準備の失敗（{name}）: {reason}")

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（あとで子ごと killpg するため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    # 起動したら True。プロセスが落ちた・時間切れなら、エラーらしい行を出して False
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return True
        except Exception:
            time.sleep(3)
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|error|Exception|Killed|failed|out of memory", l)]
    print("--- 起動しない:", url, "（全文は", log, "）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "エラーらしい行なし")
    return False

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def read_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return s5.parse_smi_mib(out)

def read_rows(path):
    # 書き足し中の raw.jsonl を読む。最後の行が書きかけなら飛ばす
    p = pathlib.Path(path)
    rows = []
    if p.exists():
        for l in p.read_text(encoding="utf-8").splitlines():
            try:
                rows.append(json.loads(l))
            except ValueError:
                pass
    return rows

GPU_NAME, CAP, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
TOTAL_MIB = int(TOTAL_MIB)
if "T4" not in GPU_NAME:
    prep_fail("GPU", f"T4 でない: {GPU_NAME}（今までの候補と比べられないので止める）")
REPO_SHA = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

# Qdrant
qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
if not wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60):
    prep_fail("Qdrant", "起動しない")

# V（GPU メモリ）: カード全体の使用量を 1 秒ごとに記録する（今までと同じ間隔）
GPU_MEM = []   # (time.time(), MiB)
def _log_gpu_mem():
    while True:
        try:
            GPU_MEM.append((time.time(), read_mib()))
        except Exception:
            pass
        time.sleep(1)
if "GPU_MEM_THREAD" not in globals():
    GPU_MEM_THREAD = threading.Thread(target=_log_gpu_mem, daemon=True)
    GPU_MEM_THREAD.start()
BASELINE_MIB = read_mib()   # 何も載っていない時の使用量

# llama-server（配布版・ubuntu・CUDA 12.8。T4 で全層 GPU を 10/23 に確かめた版）
pre = pathlib.Path("/content/llama-pre"); pre.mkdir(exist_ok=True)
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
for a in (LLAMA_ASSET, CUDART_ASSET):
    r = subprocess.run(["bash", "-c", f"set -o pipefail; curl -sfL {base}/{a} | tar xz -C {pre}"], capture_output=True, text=True)
    if r.returncode != 0:
        prep_fail("llama-server", f"{a} が取れない: {r.stderr[-300:]}")
found = sorted(pre.rglob("llama-server"))
if not found:
    prep_fail("llama-server", "配布版に llama-server が無い")
SERVER_BIN = str(found[0])
libdirs = sorted({str(p.parent) for p in pre.rglob("*.so*")})
LLAMA_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
v = subprocess.run([SERVER_BIN, "--version"], capture_output=True, text=True, env=LLAMA_ENV)
SERVER_VER = (v.stdout + v.stderr).strip().splitlines()[-2:] if (v.stdout + v.stderr).strip() else ["不明"]

# 重み（revision 固定・トークン不要）
from huggingface_hub import hf_hub_download
t = time.time()
try:
    GGUF_PATH = hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir="/content/gguf")
except Exception as e:
    prep_fail("重み", f"取れない: {repr(e)[:300]}")
GGUF_GIB = round(os.path.getsize(GGUF_PATH) / 2**30, 2)
REV = {"sha": GGUF_REV, "now": s5.hf_revision(GGUF_REPO).get("sha")}
print(GPU_NAME, "compute", CAP, "全体", TOTAL_MIB, "MiB / 起動前", BASELINE_MIB, "MiB / repo", REPO_SHA)
print("llama-server:", SERVER_VER, "/ GGUF", GGUF_FILE, GGUF_GIB, "GiB", f"（{time.time() - t:.0f} 秒）/ revision", REV)
stage("準備", "pass", gpu=GPU_NAME, gguf_gib=GGUF_GIB, repo=REPO_SHA)

In [ ]:
# 1. 索引（別プロセス・llama-server の前）。DATA_DIR・QDRANT_URL・QDRANT_COLLECTION は index.json に残り、評価側はそれを見る
WORK = OUTP / "index_work"
COLLECTION = f"r27b_{int(time.time())}"
p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/corpus_index.py", "--repo", REPO, "--work", str(WORK),
                    "--qdrant-url", "http://127.0.0.1:6333", "--collection", COLLECTION],
                   cwd=REPO, capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-3000:])
if p.returncode != 0:
    prep_fail("索引", f"rc={p.returncode}")
INDEX = json.loads((WORK / "index.json").read_text())
print(INDEX)

In [ ]:
# 2. 文脈長を決める: 8192 で起動し、思考 OFF を確かめ、62 問を LLM_MAX_TOKENS=1 で流して入力の最大を測る
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
WINDOWS = {}    # 区間名 -> (開始, 終了)。GPU の記録の切り出しに使う

def run_eval_watch(argv, env, name, log, raw_path, decide=None):
    # run_eval を流しながら raw.jsonl を見て、decide(rows) が理由を返したらそこで止める。返り値は (rc, 止めた理由)
    t0 = time.time()
    reason = None
    with open(log, "w") as f:
        proc = subprocess.Popen(argv, cwd=REPO, env=dict(os.environ, **env), stdout=f, stderr=subprocess.STDOUT,
                                start_new_session=True)
        while proc.poll() is None:
            time.sleep(WATCH_EVERY)
            if decide is not None:
                reason = decide(read_rows(raw_path))
                if reason:
                    kill(proc)
                    break
    WINDOWS[name] = (t0, time.time())
    tail = pathlib.Path(log).read_text(errors="replace").splitlines()[-4:]
    print(f"[{name}] rc={proc.returncode} {WINDOWS[name][1] - t0:.0f} 秒 / 止めた理由 {reason}\n  " + "\n  ".join(l[:200] for l in tail))
    return proc.returncode, reason

def start_server(ctx, log):
    # 起動 → GPU の確認 → 慣らしの 1 問。返り値は (proc, 起動したか, GPU の確認, 慣らしの秒)
    proc = launch(s5.llama_server_argv(SERVER_BIN, GGUF_PATH, PORT, ctx, extra=EXTRA_ARGS), str(log), LLAMA_ENV)
    up = wait(f"http://127.0.0.1:{PORT}/health", proc, str(log), 900)
    gpu = s5.llama_gpu_check(pathlib.Path(log).read_text(errors="replace"))
    warm = None
    if up and gpu["ok"]:
        body = json.dumps({"model": "local-llm", "messages": [{"role": "user", "content": "こんにちは"}],
                           "temperature": 0.1, "max_tokens": 16}).encode()
        t = time.time()
        try:
            req = urllib.request.Request(f"{BASE_URL}/chat/completions", body, {"Content-Type": "application/json"})
            urllib.request.urlopen(req, timeout=300).read()
            warm = round(time.time() - t, 1)
        except Exception as e:
            warm = f"失敗: {repr(e)[:200]}"
    print(f"[{pathlib.Path(log).name}] 起動 {up} / GPU {gpu} / 慣らし {warm}")
    return proc, up, gpu, warm

# 思考 OFF の事前確認（評価と同じ openai の呼び方。本文は出さず数だけ）
PRECHECK_PY = r'''
import json, re, sys
from openai import OpenAI
c = OpenAI(base_url=sys.argv[1], api_key="NONE")
s = c.chat.completions.create(model="local-llm", messages=[{"role": "user", "content": "日本の首都はどこですか。一文で答えてください。"}],
                              max_tokens=64, temperature=0.1, stream=True, stream_options={"include_usage": True})
content, reasoning, usage = "", "", None
for ch in s:
    if getattr(ch, "usage", None) is not None:
        usage = ch.usage
    if ch.choices:
        d = ch.choices[0].delta
        content += d.content or ""
        reasoning += ((getattr(d, "model_extra", None) or {}).get("reasoning_content") or "")
print(json.dumps({"prompt_tokens": getattr(usage, "prompt_tokens", None), "completion_tokens": getattr(usage, "completion_tokens", None),
                  "content_chars": len(content), "reasoning_chars": len(reasoning), "japanese": bool(re.search(r"[぀-ヿ]", content)),
                  "thinking_marks": bool(re.search(r"<\|?channel|<\|?think|</think>", content))}))
'''

META_COMMON = {"step": "research-27b", "gpu": GPU_NAME, "compute_cap": CAP, "model": MODEL, "model_revision": GGUF_REV,
               "runtime": f"llama.cpp {LLAMA_TAG}", "gguf": f"{GGUF_REPO}/{GGUF_FILE}", "quant": QUANT,
               "rerank": "off", "numeric_check": "off", "thinking": "off", "repo": REPO_SHA, "note": STEP_NOTE}
MEASURE = {"ctx": MEASURE_CTX}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
srv, up, gpu, warm = start_server(MEASURE_CTX, OUTP / "llama_measure.log")
if not (up and gpu["ok"]):
    kill(srv)
    prep_fail("文脈長の測定", f"8192 の llama-server が起動しない・全層が GPU に載らない（T4 は 15GB あるので容量の線とは別）: {gpu}")
pc = subprocess.run([RAG_PY, "-c", PRECHECK_PY, BASE_URL], capture_output=True, text=True)
try:
    PRECHECK = json.loads(pc.stdout.strip().splitlines()[-1])
except Exception:
    PRECHECK = {"error": (pc.stdout + pc.stderr)[-500:]}
PRECHECK["ok"] = bool((PRECHECK.get("prompt_tokens") or 0) > 0 and PRECHECK.get("content_chars", 0) > 0
                      and PRECHECK.get("reasoning_chars") == 0 and not PRECHECK.get("thinking_marks"))
print("PRECHECK_JSON " + json.dumps(PRECHECK, ensure_ascii=False))
if not PRECHECK["ok"]:
    kill(srv)
    prep_fail("思考 OFF の確認", f"通らない（入力トークン数・本文・考える文のどれか）: {PRECHECK}")
rc, _ = run_eval_watch(s5.eval_cmd(RAG_PY, OUTP / "measure", meta={**META_COMMON, "purpose": "prompt_tokens",
                                                                    "ctx": MEASURE_CTX, "llm_max_tokens": 1}),
                       s5.eval_env(INDEX["env"], BASE_URL, LLM_MAX_TOKENS="1", NUMERIC_CHECK_ENABLED=NUMERIC_CHECK),
                       "measure", str(OUTP / "measure.log"), OUTP / "measure" / "raw.jsonl")
kill(srv)
try:
    STATS = s5.prompt_token_stats(OUTP / "measure" / "raw.jsonl")   # 1 問でも取れなければ止まる
except Exception as e:
    prep_fail("文脈長の測定", repr(e)[:300])
MAX_LEN = s5.needed_max_model_len(STATS["max_prompt_tokens"])
MEASURE.update(STATS, rc=rc, needed_ctx=MAX_LEN, gpu_check=gpu, warmup_seconds=warm, precheck=PRECHECK)
print("入力の最大", STATS["max_prompt_tokens"], "トークン（", STATS["max_id"], "・", STATS["n"], "問）→ 文脈長", MAX_LEN)
stage("文脈長の測定", "pass", ctx=MAX_LEN, max_prompt_tokens=STATS["max_prompt_tokens"], max_id=STATS["max_id"])

In [ ]:
# 3. 容量と速さ（12GB・50 問）。文脈長 MAX_LEN で起動し、50 問を流しながら速さの不合格が決まったら止める
IDS_50 = s5.dataset_ids(REPO, datasets=(DATASET_50,))
IDS_N = s5.dataset_ids(REPO, datasets=(NUMERIC_12,))
assert len(IDS_50) == 50 and len(IDS_N) == 12, (len(IDS_50), len(IDS_N))
BUDGET_MIB = s5.BUDGETS_MIB[BUDGET]
LOG = OUTP / "llama_12GB.log"
OUT_50, OUT_N = OUTP / "run_50", OUTP / "run_N"
SRV = None
SPEED = {"budget": BUDGET, "budget_mib": BUDGET_MIB, "ctx": MAX_LEN}

def log_scan_since(start):
    text = LOG.read_text(errors="replace")
    whole, part = s5.scan_llama_log(text), s5.scan_llama_log(text[start:])
    # OOM と文脈あふれは起動から全部、デバッグ行の数と文脈の長さは回答の区間だけで見る
    return dict(whole, debug_active=part["debug_active"], slot_n_ctx=part["slot_n_ctx"])

s5.wait_gpu_released(read_mib, BASELINE_MIB)
t_launch = time.time()
SRV, up, gpu, warm = start_server(MAX_LEN, LOG)
WINDOWS["launch"] = (t_launch, time.time())
SPEED.update(started=up, gpu_check=gpu, warmup_seconds=warm)
if not (up and gpu["ok"]):
    scan = s5.scan_llama_log(LOG.read_text(errors="replace"))
    peak = s5.peak_in_window(GPU_MEM, t_launch, time.time())["peak_mib"]
    kill(SRV); SRV = None
    SPEED.update(peak_mib=peak, oom=scan["oom"][:5])
    if scan["oom"]:
        stage("容量", "fail", "起動でメモリ不足", peak_mib=peak)
    else:
        prep_fail("容量と速さ", f"起動しない・全層が GPU に載らない（メモリ不足の記録なし）: {gpu}")
else:
    log_start = len(LOG.read_text(errors="replace"))
    meta = {**META_COMMON, "budget": BUDGET, "budget_mib": BUDGET_MIB, "ctx": MAX_LEN, "purpose": "speed_50"}
    rc, early = run_eval_watch(s5.eval_cmd(RAG_PY, OUT_50, meta=meta, datasets=(DATASET_50,)),
                               s5.eval_env(INDEX["env"], BASE_URL, NUMERIC_CHECK_ENABLED=NUMERIC_CHECK),
                               "run_50", str(OUTP / "run_50.log"), OUT_50 / "raw.jsonl",
                               lambda rows: sg.speed_decided_fail(rows, IDS_50))
    scan = log_scan_since(log_start)
    mem = s5.gpu_mem_record(GPU_MEM, {"launch": WINDOWS["launch"], "run_50": WINDOWS["run_50"]},
                            BUDGET_MIB, TOTAL_MIB, BASELINE_MIB)
    ROWS_50 = read_rows(OUT_50 / "raw.jsonl")
    speed = sg.speed_result(ROWS_50, IDS_50)
    # 時間とトークン数だけ出す（速さで落ちたら品質は評価しないので、点の欄は出さない）
    per_q = [{k: r.get(k) for k in ("id", "first_token_latency", "total_latency", "retrieval_latency",
                                    "prompt_tokens", "completion_tokens")} | {"error": (str(r["error"])[:80] if r.get("error") else None)}
             for r in ROWS_50]
    SPEED.update(rc=rc, early_stop=early, peak_mib=mem["peak_mib"], within_budget=mem["within_budget"],
                 oom=scan["oom"][:5], overflow=scan["preempt"][:5], slot_n_ctx=scan["slot_n_ctx"],
                 decode_tok_s_p50=sg.decode_rate_median(ROWS_50), n_rows=len(ROWS_50), **speed)
    print("SPEED_JSON " + json.dumps({k: v for k, v in SPEED.items()}, ensure_ascii=False, default=str))
    print("SPEED_ROWS_JSON " + json.dumps(per_q, ensure_ascii=False))
    if mem["within_budget"] is False or scan["oom"]:
        stage("容量", "fail", f"最大 {mem['peak_mib']} MiB / 予算 {BUDGET_MIB} MiB・メモリ不足の記録 {len(scan['oom'])}",
              peak_mib=mem["peak_mib"])
    elif mem["within_budget"] is None:
        prep_fail("容量", "GPU の記録が無い区間がある（未確認）")
    else:
        stage("容量", "pass", peak_mib=mem["peak_mib"], budget_mib=BUDGET_MIB)
    if early or not speed["ok"]:
        stage("速さ", "fail", early or f"中央値 最初の文字 {speed['p1_ftl_p50']} 秒・完了 {speed['p3_total_p50']} 秒",
              p1_ftl_p50=speed["p1_ftl_p50"], p3_total_p50=speed["p3_total_p50"], answered=speed["answered"])
    else:
        stage("速さ", "pass", p1_ftl_p50=speed["p1_ftl_p50"], p3_total_p50=speed["p3_total_p50"],
              over_p3=speed["over_p3"], total_max=speed["total_max_answered"])
    if STOP:
        kill(SRV); SRV = None

In [ ]:
# 4. 品質（50 問の答えから・採点役なし）: 拒否 16 問（M8）・権限（M7）・正しさ（M6）。速さを通った時だけ
Q = {}
if STOP:
    stage("品質（M8・M7・M6）", "skipped", f"前の段で終了: {STOP['stage']}")
else:
    ds50 = [json.loads(l) for l in (pathlib.Path(REPO) / DATASET_50).read_text(encoding="utf-8").splitlines() if l.strip()]
    ref = sg.refusal_result(ROWS_50, ds50)
    answered = [r for r in ROWS_50 if not sg.answer_failed(r)]
    m7 = sum(r.get("permission_ok") is True for r in answered) / len(answered) if answered else 0.0
    m6 = sum(float(r.get("correctness") or 0.0) for r in answered) / len(IDS_50)   # 失敗・未実行は 0 点（reporter と同じ）
    Q = {"m8": ref, "m7": round(m7, 3), "m6": round(m6, 3),
         "recall_at_5": round(sum(float(r.get("recall_at_5") or 0) for r in answered) / len(IDS_50), 3)}
    print("Q_JSON " + json.dumps(Q, ensure_ascii=False))
    fails = []
    if not ref["ok"]:
        fails.append(f"M8 {ref['correct']}/{ref['n']}（外れ {ref['not_ok_ids']}）")
    if m7 < 1.0:
        fails.append(f"M7 {m7:.3f}")
    if m6 < 0.80:
        fails.append(f"M6 {m6:.3f}")
    stage("品質（M8・M7・M6）", "fail" if fails else "pass", "・".join(fails) or None, **{k: Q[k] for k in ("m7", "m6")},
          m8=f"{ref['correct']}/{ref['n']}")
    if STOP:
        kill(SRV); SRV = None

In [ ]:
# 5. 数値の境目 12 問（N）: 同じ起動のまま流し、1 問でも外れたらそこで止める
N = {}
if STOP:
    stage("数値 12 問", "skipped", f"前の段で終了: {STOP['stage']}")
else:
    log_start = len(LOG.read_text(errors="replace"))
    meta = {**META_COMMON, "budget": BUDGET, "budget_mib": BUDGET_MIB, "ctx": MAX_LEN, "purpose": "numeric_12"}
    rc, early = run_eval_watch(s5.eval_cmd(RAG_PY, OUT_N, meta=meta, datasets=(NUMERIC_12,)),
                               s5.eval_env(INDEX["env"], BASE_URL, NUMERIC_CHECK_ENABLED=NUMERIC_CHECK),
                               "run_N", str(OUTP / "run_N.log"), OUT_N / "raw.jsonl",
                               lambda rows: sg.numeric_decided_fail(rows, IDS_N))
    scan = log_scan_since(log_start)
    kill(SRV); SRV = None
    ROWS_N = read_rows(OUT_N / "raw.jsonl")
    N = dict(sg.numeric_result(ROWS_N, IDS_N), rc=rc, early_stop=early)
    # 容量は数値の区間（一番長い入力 N07 を含む）も入れて見直す
    mem = s5.gpu_mem_record(GPU_MEM, {k: WINDOWS[k] for k in ("launch", "run_50", "run_N")}, BUDGET_MIB, TOTAL_MIB, BASELINE_MIB)
    N.update(peak_mib=mem["peak_mib"], oom=scan["oom"][:5], overflow=scan["preempt"][:5],
             times=s5.numbers_only(OUT_N / "raw.jsonl")["times"] if ROWS_N else None)
    print("N_JSON " + json.dumps(N, ensure_ascii=False))
    if mem["within_budget"] is False or scan["oom"] or scan["preempt"]:
        stage("容量（数値の区間まで）", "fail", f"最大 {mem['peak_mib']} MiB・メモリ不足 {len(scan['oom'])}・文脈あふれ {len(scan['preempt'])}")
    stage("数値 12 問", "pass" if N["ok"] else "fail",
          None if N["ok"] else f"{N['correct']}/{N['n']}・{early or ''}（外れ・未実行 {N['not_ok_ids']}）",
          correct=N["correct"], n=N["n"])

In [ ]:
# 6. 回答の控え: 品質の段まで進んだ時だけ、raw.jsonl・report.md・run_meta.json を gzip+base64 でセルの出力に出す
# （根拠採点 M4-S は人が行うので回答を残す。中身は架空の社内規程に対する回答）。速さで落ちた時は出さない
import base64, gzip, io, tarfile
if any(s["stage"] == "品質（M8・M7・M6）" and s["result"] in ("pass", "fail") for s in STAGES):
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w") as tar:
        for d in (OUT_50, OUT_N):
            for name in ("raw.jsonl", "report.md", "run_meta.json"):
                if (d / name).exists():
                    tar.add(d / name, arcname=f"{d.name}/{name}")
    blob = base64.b64encode(gzip.compress(buf.getvalue())).decode()
    print(f"ANSWERS_B64_LEN {len(blob)}")
    print("ANSWERS_B64_BEGIN")
    for i in range(0, len(blob), 10000):
        print(blob[i:i + 10000])
    print("ANSWERS_B64_END")
else:
    print("品質の段まで進んでいないので回答は出さない")

In [ ]:
# 7. 権限・障害試験（G0・リランク off・12GB と同じ設定）。止める試験で LLM サーバーを落とすので最後に流す
import shutil
G0 = {"rerank": "off", "fault_timeout": FAULT_TIMEOUT, "numeric_check": NUMERIC_CHECK}
if STOP:
    stage("G0", "skipped", f"前の段で終了: {STOP['stage']}")
else:
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    G0.update(budget=BUDGET, ctx=MAX_LEN)
    srv, up, gpu, warm = start_server(MAX_LEN, OUTP / "llama_g0.log")
    G0["gpu_check"] = gpu
    if up and gpu["ok"]:
        WORK_G0 = OUTP / "g0_work"
        p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/rag_connection_check.py", "--repo", REPO,
                            "--work", str(WORK_G0), "--qdrant-url", "http://127.0.0.1:6333", "--vllm-url", BASE_URL,
                            "--qdrant-pgid", str(qdrant.pid), "--vllm-pgid", str(srv.pid),
                            "--fault-timeout", str(FAULT_TIMEOUT), "--rerank", "off",
                            "--label", f"colab-{GPU_NAME.split()[-1]}-r27b-iq3-rerank-off"],
                           cwd=REPO, capture_output=True, text=True,
                           env=dict(os.environ, NUMERIC_CHECK_ENABLED=NUMERIC_CHECK))
        (OUTP / "g0.log").write_text(p.stdout + "\n--- stderr ---\n" + p.stderr)
        print(p.stdout[-4000:], p.stderr[-3000:])
        res = WORK_G0 / "rag_check_result.json"
        G0["rc"] = p.returncode
        G0["ok"] = json.loads(res.read_text()).get("ok") if res.exists() else None
        if res.exists():
            shutil.copy(res, OUTP / "g0_result.json")
    kill(srv)
    if not (up and gpu["ok"]):
        stage("G0", "prep_failure", "G0 用の llama-server が起動しない・全層が GPU に載らない")
    elif G0["ok"] is None:
        stage("G0", "prep_failure", "結果のファイルが無い（未確認）")
    else:
        stage("G0", "pass" if G0["ok"] else "fail")
if not STOP:
    stage("根拠採点（M4-S）", "skipped", "人が採点する（ノートの外）。完了しなければ不合格")

In [ ]:
# 8. まとめ（数だけ。問・回答の本文は入れない）
SUMMARY = {"step": "research-27b", "note": STEP_NOTE, "gpu": GPU_NAME, "compute_cap": CAP, "total_mib": TOTAL_MIB,
           "baseline_mib": BASELINE_MIB, "repo": REPO_SHA, "runtime": f"llama.cpp {LLAMA_TAG}", "server_version": SERVER_VER,
           "model": MODEL, "gguf": {"repo": GGUF_REPO, "file": GGUF_FILE, "revision": REV, "gib": GGUF_GIB}, "quant": QUANT,
           "server_args": s5.llama_server_argv("llama-server", GGUF_FILE, PORT, MAX_LEN, extra=EXTRA_ARGS)[1:],
           "rerank": "off", "numeric_check": NUMERIC_CHECK, "thinking": "off", "budget": BUDGET, "budget_mib": BUDGET_MIB,
           "measure": MEASURE, "speed": SPEED, "quality": Q, "numeric": N, "g0": G0, "stages": STAGES,
           "stopped_at": STOP, "result": ("fail" if STOP and STOP["result"] == "fail" else
                                          "prep_failure" if STOP else "pass_pending_m4s")}
(OUTP / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2, default=str) + "\n")
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False, default=str))
for s in STAGES:
    print(f"{s['stage']}: {s['result']}" + (f"（{s['reason']}）" if s.get("reason") else ""))

In [ ]:
# 後片付け: サーバーを落とし、結果を zip にする（回答の本文を含む。リポジトリに入れるのは summary.json と run_meta.json だけ）
kill(SRV)
kill(qdrant)
!cd /content && zip -qr r27b_results.zip r27b -x 'r27b/index_work/*' 'r27b/g0_work/*'
!ls -la /content/r27b_results.zip